# Computation of one-electron integrals 

In the following section we develop python functions for the efficient calculation of one-electron integrals over Cartesian Gaussian basis functions. We will focus on the overlap integral, which is the simplest of the one-electron integrals, and then we will extend the approach to other types of integrals (kinetic, nuclear attraction, etc.) in subsequent sections. 

In [ ]:
import numpy as np
from numba import njit
@njit(fastmath=True)
def overlap_shell_pair(exp_a: np.ndarray, coeff_a: np.ndarray, norm_a: np.ndarray, center_a: np.ndarray,
                       exp_b: np.ndarray, coeff_b: np.ndarray, norm_b: np.ndarray, center_b: np.ndarray, ijk_lmn: np.ndarray,) -> np.ndarray:
    dim_a = norm_a.shape[1]
    dim_b = norm_b.shape[1]
    out = np.zeros((dim_a, dim_b), dtype=np.float64)

    Dx = center_a[0] - center_b[0]
    Dy = center_a[1] - center_b[1]
    Dz = center_a[2] - center_b[2]
    D2 = Dx * Dx + Dy * Dy + Dz * Dz

    for idx in range(ijk_lmn.shape[0]):
        i, j, k, l, m, n = ijk_lmn[idx]
        ia = idx // dim_b
        ib = idx - ia * dim_b

        val = 0.0
        for pa in range(exp_a.shape[0]):
            alpha_a = exp_a[pa]
            ca = coeff_a[pa] * norm_a[pa, ia]
            for pb in range(exp_b.shape[0]):
                alpha_b = exp_b[pb]
                cb = coeff_b[pb] * norm_b[pb, ib]
                ab_sum = alpha_a + alpha_b
                ab_product = alpha_a * alpha_b
                val += ca * cb * S(i,j,k,l,m,n,Dx,Dy,Dz,D2,ab_sum,ab_product,alpha_a,alpha_b,)
        out[ia, ib] = val

    return out


## How does `overlap_shell_pair` work

This function computes **overlap block** between two Gaussian shells `a` and `b`. Here, Shell `a` contains `dim_a` Cartesian functions and Shell `b` contains `dim_b` Cartesian functions. The result is a matrix `out` of shape `(dim_a, dim_b)`, where each entry is one contracted overlap integral between one function from shell `a` and one from shell `b`. The function is decorated with `@njit(fastmath=True)`, which means that it is compiled with Numba for performance, and it allows for fast math optimizations. The resulting overlap block is used in the assembly of the full AO overlap matrix by inserting it into the appropriate position corresponding to the basis functions of shells `a` and `b`.

Each contracted Cartesian basis function is a linear combination of primitive Gaussians.  
So each overlap element is a **double sum over primitive pairs**:

$$
S_{\mu\nu}^{(ab)}
=
\sum_{p=1}^{N_a}\sum_{q=1}^{N_b}
\Big(d_p^{(a)} N_{p\mu}^{(a)}\Big)
\Big(d_q^{(b)} N_{q\nu}^{(b)}\Big)
\,S_{\text{prim}}(\mu,\nu,p,q),
$$

where:

- $d$ = contraction coefficients (`coeff_a`, `coeff_b`)
- $N$ = primitive normalization factors (`norm_a`, `norm_b`)
- $S_{\text{prim}}$ = primitive Cartesian overlap value (computed by `S(...)`)

The angular powers needed by `S(...)` are taken from `ijk_lmn`.


## Assembly of the full AO overlap matrix
The full AO overlap matrix is assembled by looping over all pairs of shells in the basis set, computing the overlap block for each pair using `overlap_shell_pair`, and inserting it into the appropriate position in the full matrix. The resulting full AO overlap matrix is symmetric, so we only need to compute the lower triangle and then mirror it to the upper triangle.

`overlap_matrix_driver` is the routine that takes all prepacked shell data and assembles the full atomic-orbital overlap matrix in one pass. It starts by allocating a square matrix S_ao with size n_ao x n_ao, where n_ao is the total number of AO basis functions in the molecule. From there, it loops over a list of shell-pair indices. At each loop step p, it reads the two shell ids a = pair_a[p] and b = pair_b[p]. These two arrays already encode which shell pairs must be evaluated.

To place a shell block into the global matrix, the function uses offsets. For shell a, the AO rows are a0:a1 with a0 = offsets[a] and a1 = offsets[a+1]; for shell b, the AO columns are b0:b1. 

The actual numerical block is computed by calling overlap_shell_pair(...). That call receives all data belonging to shell a and shell b—primitive exponents, contraction coefficients, normalization factors, centers, and the angular-index table for this specific pair (ijk_data[p]). The returned block has exactly the shape needed for insertion: (a1-a0, b1-b0). The driver writes it into S_ao[a0:a1, b0:b1].

Because overlap matrices are symmetric, the function does not recompute the transposed block. If a != b, it simply mirrors the result by assigning block.T into the opposite position S_ao[b0:b1, a0:a1]. 

The @njit(fastmath=True) decorator means this whole assembly loop is compiled by Numba to machine code. This removes Python-loop overhead during matrix assembly and lets the driver run at compiled speed while repeatedly calling the compiled shell kernel. 

In [ ]:
@njit(fastmath=True)
def overlap_matrix_driver(n_ao: int, pair_a: np.ndarray, pair_b: np.ndarray, 
                          offsets: np.ndarray, exp_data, coeff_data, norm_data, center_data, ijk_data,) -> np.ndarray:
    S_ao = np.zeros((n_ao, n_ao), dtype=np.float64)

    for p in range(pair_a.shape[0]):
        a = pair_a[p]
        b = pair_b[p]

        a0 = offsets[a]
        a1 = offsets[a + 1]
        b0 = offsets[b]
        b1 = offsets[b + 1]

        block = overlap_shell_pair(exp_data[a],coeff_data[a],norm_data[a],center_data[a],
                                   exp_data[b],coeff_data[b],norm_data[b],center_data[b],ijk_data[p])

        S_ao[a0:a1, b0:b1] = block
        if a != b:
            S_ao[b0:b1, a0:a1] = block.T

    return S_ao
    